In [2]:
import networkx as nx
import numpy as np
from scipy.linalg import expm

import pandas as pd

In [ ]:
def diffusion_kernel(G, beta=0.1, normalized=True):

    if normalized:
        L = nx.normalized_laplacian_matrix(G).todense()
    else:
        L = nx.laplacian_matrix(G).todense()
    
    K = expm(-beta * L)

    return np.array(K)

In [4]:
file_path = '/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_full_2019.txt'
G = nx.read_edgelist(file_path, nodetype=str, create_using=nx.Graph())

In [5]:
K = diffusion_kernel(G, beta=0.5)

In [6]:
K.shape

(19576, 19576)

In [10]:
nodes = list(G.nodes())
nodes[:2]

['9606.ENSP00000000233', '9606.ENSP00000263431']

In [11]:


i = nodes.index('9606.ENSP00000000233')
j = nodes.index('9606.ENSP00000263431')
print("Similarity:", K[i, j])


Similarity: 0.00016534348491191748


In [29]:
sim_matrix = np.array([
    [1.0, 0.8, 0.75, 0.3, 0.6],
    [0.8, 1.0, 0.9, 0.4, 0.7],
    [0.75, 0.9, 1.0, 0.5, 0.65],
    [0.3, 0.4, 0.5, 1.0, 0.2],
    [0.6, 0.7, 0.65, 0.2, 1.0]
])

In [32]:
import pandas as pd
import numpy as np

# Sample inputs
sim_matrix = np.array([
    [1.0, 0.8, 0.75, 0.3, 0.6],
    [0.8, 1.0, 0.9, 0.4, 0.7],
    [0.75, 0.9, 1.0, 0.5, 0.65],
    [0.3, 0.4, 0.5, 1.0, 0.2],
    [0.6, 0.7, 0.65, 0.2, 1.0]
])
sample_names = ['S0', 'S1', 'S2', 'S3', 'S4']

# Inputs for merging and deletion
merge_groups = [['S1', 'S2']]
delete_list = ['S3']

# Step 1: Convert to tidy format
df = pd.DataFrame(sim_matrix, index=sample_names, columns=sample_names)
tidy = df.stack().reset_index()
tidy.columns = ['Sample_i', 'Sample_j', 'Similarity']
tidy = tidy[tidy['Sample_i'] < tidy['Sample_j']]  # remove duplicates

# Step 2: Precompute all pair similarities
# For quick lookup, use frozen set of pair as key
pair_sim = {
    frozenset([i, j]): s for i, j, s in tidy.itertuples(index=False)
}

# Step 3: Build new groups
sample_to_group = {}
new_names = []
for idx, group in enumerate(merge_groups):
    new_name = '_'.join(sorted(group))
    new_names.append(new_name)
    for s in group:
        sample_to_group[s] = new_name

# Samples not in any merge group or deletion
all_samples = set(sample_names)
merged_samples = set(sample_to_group.keys())
kept_samples = sorted(all_samples - merged_samples - set(delete_list))

# Step 4: Build final sample list and group mapping
final_samples = new_names + kept_samples
group_map = {name: [name] for name in kept_samples}
for group in merge_groups:
    group_name = '_'.join(sorted(group))
    group_map[group_name] = group

# Step 5: Compute average similarity between groups
records = []
for i, group_i in enumerate(final_samples):
    for j in range(i, len(final_samples)):
        group_j = final_samples[j]
        members_i = group_map[group_i]
        members_j = group_map[group_j]
        
        # Compute all pairwise similarities
        sims = []
        for a in members_i:
            for b in members_j:
                if a != b:
                    key = frozenset([a, b])
                    if key in pair_sim:
                        sims.append(pair_sim[key])
        # Self-similarity
        sim_val = 1.0 if group_i == group_j else np.mean(sims)
        records.append((group_i, group_j, sim_val))
        if group_i != group_j:
            records.append((group_j, group_i, sim_val))

# Step 6: Build final tidy and square matrix
new_tidy = pd.DataFrame(records, columns=['Sample_i', 'Sample_j', 'Similarity'])
new_matrix = new_tidy.pivot(index='Sample_i', columns='Sample_j', values='Similarity')
new_matrix = new_matrix.reindex(index=final_samples, columns=final_samples, fill_value=1.0)

# Output
print("\nTidy format after merge and deletion:")
print(new_tidy.sort_values(by=['Sample_i', 'Sample_j']).reset_index(drop=True))

print("\nNew similarity matrix:")
print(np.round(new_matrix, 3))



Tidy format after merge and deletion:
  Sample_i Sample_j  Similarity
0       S0       S0       1.000
1       S0    S1_S2       0.775
2       S0       S4       0.600
3    S1_S2       S0       0.775
4    S1_S2    S1_S2       1.000
5    S1_S2       S4       0.675
6       S4       S0       0.600
7       S4    S1_S2       0.675
8       S4       S4       1.000

New similarity matrix:
Sample_j  S1_S2     S0     S4
Sample_i                     
S1_S2     1.000  0.775  0.675
S0        0.775  1.000  0.600
S4        0.675  0.600  1.000


In [ ]:
sim_matrix = np.array([
    [1.0, 0.8, 0.75, 0.3, 0.6],
    [0.8, 1.0, 0.9, 0.4, 0.7],
    [0.75, 0.9, 1.0, 0.5, 0.65],
    [0.3, 0.4, 0.5, 1.0, 0.2],
    [0.6, 0.7, 0.65, 0.2, 1.0]
])

In [8]:
import pickle
import os

In [3]:
with open('/itf-fi-ml/shared/users/ziyuzh/svm/results/dw_auc_norm_test/2019/path_save.pkl', 'rb') as f:
    kernels_all_dict = pickle.load(f)

In [ ]:
df_path = '/itf-fi-ml/shared/users/ziyuzh/svm/results/df/2019'
df_dict = dict()
for file in os.listdir(df_path):
    if 'uniport_diffusion_K' in file:
        temp_list = []
        key = file.split('_')[-1][:-4]
        temp_list.append(os.path.join(df_path,file))
        temp_list.append(os.path.join(df_path,'uniport_difussion_logK_'+file.split('_')[-1]))
        df_dict[key] = temp_list

In [ ]:
kernels_all_dict['diffusion_kernel'] = df_dict

['/itf-fi-ml/shared/users/ziyuzh/svm/results/df/test/diffusion_K_0.5.pkl',
 '/itf-fi-ml/shared/users/ziyuzh/svm/results/df/test/uniport_difussion_logK_0.5.pkl']

In [7]:
kernels_all_dict

{'uniport_bio': {2: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/dw_auc_norm_test/2019/uniport_bio_K_2_0.5427586367192363.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/dw_auc_norm_test/2019/uniport_bio_logK_2_0.5427586367192363.pkl'],
  4: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/dw_auc_norm_test/2019/uniport_bio_K_4_0.27137931835961815.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/dw_auc_norm_test/2019/uniport_bio_logK_4_0.27137931835961815.pkl'],
  8: ['/itf-fi-ml/shared/users/ziyuzh/svm/results/dw_auc_norm_test/2019/uniport_bio_K_8_0.13568965917980907.pkl',
   '/itf-fi-ml/shared/users/ziyuzh/svm/results/dw_auc_norm_test/2019/uniport_bio_logK_8_0.13568965917980907.pkl']}}

In [10]:
X_k_path = '/itf-fi-ml/shared/users/ziyuzh/svm/results/dw_auc/2019/uniport_bio_K_2_0.05853505840227169.pkl'
X_logk_path = '/itf-fi-ml/shared/users/ziyuzh/svm/results/dw_auc/2019/uniport_bio_logK_2_0.05853505840227169.pkl'

In [ ]:
with open(X_k_path, 'rb') as f:
    X_k = pickle.load(f)
with open(X_logk_path, 'rb') as f:
    X_logk = pickle.load(f)

ks = [X_k]
logks = [X_logk]

def normalize_kernel(K):
    diag = np.sqrt(np.diag(K))
    diag[diag == 0] = 1e-8  # Avoid division by zero
    return K / (diag[:, None] * diag[None, :])

In [13]:
K_linear_fused = np.mean(ks, axis=0)

np.array_equal(K_linear_fused, X_k)

True

In [16]:
K_linear_fused = 0.5 * (K_linear_fused + K_linear_fused.T)
np.array_equal(K_linear_fused, X_k), np.allclose(K_linear_fused, X_k)

(False, True)

In [17]:
K_linear_fused = normalize_kernel(K_linear_fused)
np.array_equal(K_linear_fused, X_k), np.allclose(K_linear_fused, X_k)

(False, True)

In [ ]:
logk_avg = np.mean(logks, axis=0)
eigenvalues, eigenvectors = np.linalg.eigh(logk_avg)
eigenvalues = np.clip(eigenvalues, -50, 50)  # Prevent overflow
K_geo_mean = eigenvectors @ np.diag(np.exp(eigenvalues)) @ eigenvectors.T
np.array_equal(K_geo_mean, X_k), np.allclose(K_geo_mean, X_k)

(False, True)

In [20]:
X_k_sym = 0.5 * (X_k + X_k.T)
K_linear_fused = np.mean([X_k_sym], axis=0)
K_linear_fused = 0.5 * (K_linear_fused + K_linear_fused.T)
K_linear_fused = normalize_kernel(K_linear_fused)
np.array_equal(K_linear_fused, X_k_sym), np.allclose(K_linear_fused, X_k_sym)

(True, True)

In [ ]:
def process_kernel(args):
    K= args

    eigenvalues, eigenvectors = np.linalg.eigh(K)
    eigenvalues = np.clip(eigenvalues, 1e-12, None)  # Avoid log(0)
    K_log = eigenvectors @ np.diag(np.log(eigenvalues)) @ eigenvectors.T
    K_log = 0.5 * (K_log + K_log.T)

    return K_log

logm_k_sym = process_kernel(X_k_sym)

logk_avg = np.mean([logm_k_sym], axis=0)
eigenvalues, eigenvectors = np.linalg.eigh(logk_avg)
eigenvalues = np.clip(eigenvalues, -50, 50)  # Prevent overflow
K_geo_mean = eigenvectors @ np.diag(np.exp(eigenvalues)) @ eigenvectors.T
np.array_equal(K_geo_mean, X_k_sym), np.allclose(K_geo_mean, X_k_sym)

(False, True)

In [24]:
# Load the same kernel used in select_gamma_ratio
with open('/itf-fi-ml/shared/users/ziyuzh/svm/results/dw_auc_norm_test/2019/uniport_bio_K_2_0.5427586367192363.pkl', 'rb') as f:
    X_k = pickle.load(f)

K_linear_fused = np.mean([X_k], axis=0)
K_linear_fused = 0.5 * (K_linear_fused + K_linear_fused.T)
K_linear_fused = normalize_kernel(K_linear_fused)

# Compare to your fused version
np.allclose(X_k, K_linear_fused) , np.allclose(normalize_kernel(X_k), K_linear_fused)  # Might be closer to True


(True, True)

In [27]:
import pandas as pd
import numpy as np

# Create a DataFrame with 4 columns and 100 rows of random values between 0 and 1
df = pd.DataFrame(np.random.rand(100, 4), columns=['A', 'B', 'C', 'D'])

In [28]:
df.sample(n=10, replace=True, random_state=42)

,A,B,C,D
51,0.462025,0.275829,0.112412,0.238344
92,0.745503,0.293508,0.102453,0.494458
14,0.983508,0.363192,0.966372,0.799653
71,0.095200,0.610888,0.283372,0.885774
60,0.676524,0.186099,0.905720,0.158129
20,0.473807,0.596660,0.633599,0.276020
82,0.258661,0.125689,0.488922,0.896569
86,0.113321,0.263151,0.922083,0.336097
74,0.215723,0.524986,0.683844,0.462290
74,0.215723,0.524986,0.683844,0.462290


In [29]:
df.sample(n=10, replace=True, random_state=42)

,A,B,C,D
51,0.462025,0.275829,0.112412,0.238344
92,0.745503,0.293508,0.102453,0.494458
14,0.983508,0.363192,0.966372,0.799653
71,0.095200,0.610888,0.283372,0.885774
60,0.676524,0.186099,0.905720,0.158129
20,0.473807,0.596660,0.633599,0.276020
82,0.258661,0.125689,0.488922,0.896569
86,0.113321,0.263151,0.922083,0.336097
74,0.215723,0.524986,0.683844,0.462290
74,0.215723,0.524986,0.683844,0.462290
